In [26]:
%pip install requests pyreadr

Note: you may need to restart the kernel to use updated packages.


In [27]:
from pathlib import Path
import requests

# Project root: notebook is inside the notebooks folder
project_root = Path.cwd().parent

# Create raw data folder
raw_folder = project_root / "data" / "raw"
raw_folder.mkdir(parents=True, exist_ok=True)

# Dataset URL
url = (
    "https://raw.githubusercontent.com/cran/"
    "denguedatahub/master/data/srilanka_weekly_data.rda"
)

rda_path = raw_folder / "srilanka_weekly_data.rda"

# Download
response = requests.get(url, timeout=60)
response.raise_for_status()

rda_path.write_bytes(response.content)

print("Download completed!")
print("Saved location:", rda_path)
print("File size:", round(rda_path.stat().st_size / 1024, 2), "KB")

Download completed!
Saved location: c:\Users\Nethmi\Documents\GitHub\RiskWeave\data\raw\srilanka_weekly_data.rda
File size: 37.68 KB


In [28]:
import pyreadr
import pandas as pd

# Read the downloaded RDA file
rda_path = raw_folder / "srilanka_weekly_data.rda"

result = pyreadr.read_r(str(rda_path))

# Display available objects
print("Objects found:")
print(result.keys())

Objects found:
odict_keys(['srilanka_weekly_data'])


In [29]:
dengue = result["srilanka_weekly_data"]

print("Dataset shape:", dengue.shape)

display(dengue.head())
display(dengue.tail())

Dataset shape: (25766, 6)


,year,week,start.date,end.date,district,cases
0,2006.0,52.0,12/23/2006,12/29/2006,Colombo,71.0
1,2006.0,52.0,12/23/2006,12/29/2006,Gampaha,12.0
2,2006.0,52.0,12/23/2006,12/29/2006,Kalutara,12.0
3,2006.0,52.0,12/23/2006,12/29/2006,Kandy,20.0
4,2006.0,52.0,12/23/2006,12/29/2006,Matale,4.0


,year,week,start.date,end.date,district,cases
25761,2025.0,52.0,12/13/2025,12/19/2025,Badulla,25.0
25762,2025.0,52.0,12/13/2025,12/19/2025,Monaragala,24.0
25763,2025.0,52.0,12/13/2025,12/19/2025,Ratnapura,60.0
25764,2025.0,52.0,12/13/2025,12/19/2025,Kegalle,37.0
25765,2025.0,52.0,12/13/2025,12/19/2025,Kalmune,8.0


In [30]:
print("Number of unique areas:", dengue["district"].nunique())

print("\nAll geographic areas:")
print(sorted(dengue["district"].dropna().unique()))

Number of unique areas: 26

All geographic areas:
['Ampara', 'Anuradhapura', 'Badulla', 'Batticaloa', 'Colombo', 'Galle', 'Gampaha', 'Hambanthota', 'Jaffna', 'Kalmune', 'Kalutara', 'Kandy', 'Kegalle', 'Kilinochchi', 'Kurunegala', 'Mannar', 'Matale', 'Matara', 'Monaragala', 'Mullaitivu', 'NuwaraEliya', 'Polonnaruwa', 'Puttalam', 'Ratnapura', 'Trincomalee', 'Vavuniya']


In [31]:
dengue["start.date"] = pd.to_datetime(
    dengue["start.date"], format="%m/%d/%Y", errors="coerce"
)

dengue["end.date"] = pd.to_datetime(
    dengue["end.date"], format="%m/%d/%Y", errors="coerce"
)

print("Earliest reporting date:", dengue["start.date"].min())
print("Latest reporting date:", dengue["end.date"].max())

print("\nMissing start dates:", dengue["start.date"].isna().sum())
print("Missing end dates:", dengue["end.date"].isna().sum())

Earliest reporting date: 2006-12-23 00:00:00
Latest reporting date: 2025-12-19 00:00:00

Missing start dates: 0
Missing end dates: 0


In [32]:
# Retrieve a fresh copy from the original RDA file
dengue_raw = result["srilanka_weekly_data"].copy()

# Save the original extracted data
raw_csv_path = raw_folder / "dengue_weekly_historical_raw.csv"

dengue_raw.to_csv(raw_csv_path, index=False)

print("Original extracted dataset saved.")
print("Location:", raw_csv_path)
print("Shape:", dengue_raw.shape)

Original extracted dataset saved.
Location: c:\Users\Nethmi\Documents\GitHub\RiskWeave\data\raw\dengue_weekly_historical_raw.csv
Shape: (25766, 6)


In [33]:
yearly_summary = (
    dengue.groupby("year")
    .agg(
        total_records=("cases", "count"),
        number_of_areas=("district", "nunique"),
        number_of_weeks=("week", "nunique"),
        first_week=("start.date", "min"),
        last_week=("end.date", "max")
    )
    .reset_index()
)

display(yearly_summary)

,year,total_records,number_of_areas,number_of_weeks,first_week,last_week
0,2006.0,26,26,1,2006-12-23,2006-12-29
1,2007.0,1352,26,52,2006-12-30,2007-12-28
2,2008.0,1352,26,52,2007-12-29,2008-12-26
3,2009.0,1378,26,53,2008-12-27,2010-01-01
4,2010.0,1352,26,52,2010-01-02,2010-12-31
5,2011.0,1352,26,52,2011-01-01,2011-12-30
6,2012.0,1352,26,52,2011-12-31,2012-12-28
7,2013.0,1352,26,52,2012-12-29,2013-12-27
8,2014.0,1352,26,52,2013-12-28,2014-12-26
9,2015.0,1352,26,52,2014-12-27,2015-12-25


In [34]:
print("Missing values:")
display(dengue.isnull().sum())

print("Exact duplicate rows:", dengue.duplicated().sum())

print(
    "Duplicate district-week records:",
    dengue.duplicated(
        subset=["year", "week", "district"]
    ).sum()
)

Missing values:


year          0
week          0
start.date    0
end.date      0
district      0
cases         0
dtype: int64

Exact duplicate rows: 0
Duplicate district-week records: 0


In [35]:
dengue["week_length_days"] = (
    dengue["end.date"] - dengue["start.date"]
).dt.days + 1

print("Reporting period lengths:")
display(
    dengue["week_length_days"]
    .value_counts()
    .sort_index()
)

print("Records with periods other than 7 days:")
display(
    dengue[dengue["week_length_days"] != 7].head(20)
)

Reporting period lengths:


week_length_days
6       26
7    25714
8       26
Name: count, dtype: int64

Records with periods other than 7 days:


,year,week,start.date,end.date,district,cases,week_length_days
3146,2009.0,17.0,2009-04-18,2009-04-25,Colombo,57.0,8
3147,2009.0,17.0,2009-04-18,2009-04-25,Gampaha,17.0,8
3148,2009.0,17.0,2009-04-18,2009-04-25,Kalutara,15.0,8
3149,2009.0,17.0,2009-04-18,2009-04-25,Kandy,73.0,8
3150,2009.0,17.0,2009-04-18,2009-04-25,Matale,18.0,8
3151,2009.0,17.0,2009-04-18,2009-04-25,NuwaraEliya,5.0,8
3152,2009.0,17.0,2009-04-18,2009-04-25,Galle,2.0,8
3153,2009.0,17.0,2009-04-18,2009-04-25,Hambanthota,28.0,8
3154,2009.0,17.0,2009-04-18,2009-04-25,Matara,5.0,8
3155,2009.0,17.0,2009-04-18,2009-04-25,Jaffna,0.0,8


In [36]:
# Select records with unusual reporting-period lengths
unusual_weeks = dengue[
    dengue["week_length_days"] != 7
].copy()

# Group by reporting year, week and actual dates
unusual_summary = (
    unusual_weeks.groupby(
        ["year", "week", "start.date", "end.date", "week_length_days"]
    )
    .agg(
        number_of_areas=("district", "nunique"),
        total_reported_cases=("cases", "sum")
    )
    .reset_index()
    .sort_values(["year", "week"])
)

display(unusual_summary)

,year,week,start.date,end.date,week_length_days,number_of_areas,total_reported_cases
0,2009.0,17.0,2009-04-18,2009-04-25,8,26,334.0
1,2009.0,22.0,2009-05-24,2009-05-29,6,26,1135.0


In [37]:
for _, row in unusual_summary.iterrows():

    year = row["year"]
    week = row["week"]

    print(f"\nReporting year: {year}, Week: {week}")

    nearby = dengue[
        (dengue["year"] == year) &
        (dengue["week"].between(week - 1, week + 1))
    ]

    display(
        nearby[
            ["year", "week", "start.date", "end.date"]
        ]
        .drop_duplicates()
        .sort_values("week")
    )


Reporting year: 2009.0, Week: 17.0


,year,week,start.date,end.date
3120,2009.0,16.0,2009-04-11,2009-04-17
3146,2009.0,17.0,2009-04-18,2009-04-25
3172,2009.0,18.0,2009-04-26,2009-05-02



Reporting year: 2009.0, Week: 22.0


,year,week,start.date,end.date
3250,2009.0,21.0,2009-05-17,2009-05-23
3276,2009.0,22.0,2009-05-24,2009-05-29
3302,2009.0,23.0,2009-05-30,2009-06-05


In [38]:
print("Missing values:")
display(dengue.isnull().sum())

print("Exact duplicate rows:", dengue.duplicated().sum())

print(
    "Duplicate district-week records:",
    dengue.duplicated(
        subset=["year", "week", "district"]
    ).sum()
)

Missing values:


year                0
week                0
start.date          0
end.date            0
district            0
cases               0
week_length_days    0
dtype: int64

Exact duplicate rows: 0
Duplicate district-week records: 0


In [39]:
# Documented reporting-date discrepancies
date_corrections = pd.DataFrame([
    {
        "year": 2009,
        "week": 17,
        "original_start": "2009-04-18",
        "original_end": "2009-04-25",
        "verified_start": "2009-04-18",
        "verified_end": "2009-04-24",
        "reason": "Official Epidemiology Unit reporting calendar"
    },
    {
        "year": 2009,
        "week": 22,
        "original_start": "2009-05-24",
        "original_end": "2009-05-29",
        "verified_start": "2009-05-23",
        "verified_end": "2009-05-29",
        "reason": "Official Epidemiology Unit reporting calendar"
    }
])

date_corrections.to_csv(
    raw_folder / "dengue_date_correction_audit.csv",
    index=False
)

display(date_corrections)

,year,week,original_start,original_end,verified_start,verified_end,reason
0,2009,17,2009-04-18,2009-04-25,2009-04-18,2009-04-24,Official Epidemiology Unit reporting calendar
1,2009,22,2009-05-24,2009-05-29,2009-05-23,2009-05-29,Official Epidemiology Unit reporting calendar


In [40]:
# Check how Kalmune appears across the years
kalmune_data = dengue[
    dengue["district"].str.lower().isin(
        ["kalmune", "kalmunai"]
    )
]

display(
    kalmune_data.groupby("year").agg(
        records=("cases", "count"),
        first_date=("start.date", "min"),
        last_date=("end.date", "max"),
        total_cases=("cases", "sum")
    ).reset_index()
)

,year,records,first_date,last_date,total_cases
0,2006.0,1,2006-12-23,2006-12-29,0.0
1,2007.0,52,2006-12-30,2007-12-28,3.0
2,2008.0,52,2007-12-29,2008-12-26,32.0
3,2009.0,53,2008-12-27,2010-01-01,240.0
4,2010.0,52,2010-01-02,2010-12-31,399.0
5,2011.0,52,2011-01-01,2011-12-30,66.0
6,2012.0,52,2011-12-31,2012-12-28,138.0
7,2013.0,52,2012-12-29,2013-12-27,256.0
8,2014.0,52,2013-12-28,2014-12-26,387.0
9,2015.0,52,2014-12-27,2015-12-25,392.0


In [41]:
# Compare Ampara and Kalmune reporting records

comparison = dengue[
    dengue["district"].isin(["Ampara", "Kalmune"])
].copy()

comparison_summary = (
    comparison.groupby(["year", "district"])
    .agg(
        number_of_records=("cases", "count"),
        total_cases=("cases", "sum"),
        first_date=("start.date", "min"),
        last_date=("end.date", "max")
    )
    .reset_index()
)

display(comparison_summary)

,year,district,number_of_records,total_cases,first_date,last_date
0,2006.0,Ampara,1,0.0,2006-12-23,2006-12-29
1,2006.0,Kalmune,1,0.0,2006-12-23,2006-12-29
2,2007.0,Ampara,52,5.0,2006-12-30,2007-12-28
3,2007.0,Kalmune,52,3.0,2006-12-30,2007-12-28
4,2008.0,Ampara,52,14.0,2007-12-29,2008-12-26
5,2008.0,Kalmune,52,32.0,2007-12-29,2008-12-26
6,2009.0,Ampara,53,186.0,2008-12-27,2010-01-01
7,2009.0,Kalmune,53,240.0,2008-12-27,2010-01-01
8,2010.0,Ampara,52,85.0,2010-01-02,2010-12-31
9,2010.0,Kalmune,52,399.0,2010-01-02,2010-12-31


In [42]:
# Compare the reporting dates of Ampara and Kalmune

area_week_comparison = comparison.pivot_table(
    index=["year", "week", "start.date", "end.date"],
    columns="district",
    values="cases",
    aggfunc="count"
).reset_index()

print("Reporting periods with both areas:")
display(
    area_week_comparison[
        area_week_comparison["Ampara"].notna() &
        area_week_comparison["Kalmune"].notna()
    ].head(10)
)

print("Reporting periods missing one of the areas:")
display(
    area_week_comparison[
        area_week_comparison["Ampara"].isna() |
        area_week_comparison["Kalmune"].isna()
    ].head(20)
)

Reporting periods with both areas:


district,year,week,start.date,end.date,Ampara,Kalmune
0,2006.0,52.0,2006-12-23,2006-12-29,1,1
1,2007.0,1.0,2006-12-30,2007-01-05,1,1
2,2007.0,2.0,2007-01-06,2007-01-12,1,1
3,2007.0,3.0,2007-01-13,2007-01-19,1,1
4,2007.0,4.0,2007-01-20,2007-01-26,1,1
5,2007.0,5.0,2007-01-27,2007-02-02,1,1
6,2007.0,6.0,2007-02-03,2007-02-09,1,1
7,2007.0,7.0,2007-02-10,2007-02-16,1,1
8,2007.0,8.0,2007-02-17,2007-02-23,1,1
9,2007.0,9.0,2007-02-24,2007-03-02,1,1


Reporting periods missing one of the areas:


district,year,week,start.date,end.date,Ampara,Kalmune


In [43]:
# Extract reporting periods for each area
ampara_weeks = (
    comparison[comparison["district"] == "Ampara"]
    [["year", "week", "start.date", "end.date"]]
    .drop_duplicates()
)

kalmune_weeks = (
    comparison[comparison["district"] == "Kalmune"]
    [["year", "week", "start.date", "end.date"]]
    .drop_duplicates()
)

# Compare the complete sets of reporting periods
alignment_check = ampara_weeks.merge(
    kalmune_weeks,
    on=["year", "week", "start.date", "end.date"],
    how="outer",
    indicator=True
)

print("Reporting period alignment:")
print(alignment_check["_merge"].value_counts())

print("\nPeriods that do not match:")
display(
    alignment_check[alignment_check["_merge"] != "both"]
)

Reporting period alignment:
_merge
both          991
left_only       0
right_only      0
Name: count, dtype: int64

Periods that do not match:


,year,week,start.date,end.date,_merge


In [44]:
# Create a working copy
dengue_clean = dengue.copy()

# Standardise column names
dengue_clean = dengue_clean.rename(columns={
    "start.date": "week_start",
    "end.date": "week_end",
    "district": "area",
    "cases": "dengue_cases"
})

# Convert numeric columns
dengue_clean["year"] = dengue_clean["year"].astype("int64")
dengue_clean["week"] = dengue_clean["week"].astype("int64")
dengue_clean["dengue_cases"] = (
    dengue_clean["dengue_cases"].astype("int64")
)

# Standardise area names by removing surrounding whitespace
dengue_clean["area"] = dengue_clean["area"].str.strip()

# Calculate actual reporting-period length
dengue_clean["reporting_period_days"] = (
    dengue_clean["week_end"] -
    dengue_clean["week_start"]
).dt.days + 1

# Sort records
dengue_clean = dengue_clean.sort_values(
    ["week_start", "area"]
).reset_index(drop=True)

display(dengue_clean.head())

,year,week,week_start,week_end,area,dengue_cases,week_length_days,reporting_period_days
0,2006,52,2006-12-23,2006-12-29,Ampara,0,7,7
1,2006,52,2006-12-23,2006-12-29,Anuradhapura,4,7,7
2,2006,52,2006-12-23,2006-12-29,Badulla,2,7,7
3,2006,52,2006-12-23,2006-12-29,Batticaloa,0,7,7
4,2006,52,2006-12-23,2006-12-29,Colombo,71,7,7


In [45]:
print("Dataset shape:", dengue_clean.shape)

print("\nUnique areas:", dengue_clean["area"].nunique())

print("\nDate range:")
print(dengue_clean["week_start"].min())
print(dengue_clean["week_end"].max())

print("\nMissing values:")
display(dengue_clean.isnull().sum())

print(
    "\nDuplicate area-week records:",
    dengue_clean.duplicated(
        subset=["year", "week", "area"]
    ).sum()
)

print("\nReporting-period lengths:")
display(
    dengue_clean["reporting_period_days"]
    .value_counts()
    .sort_index()
)

Dataset shape: (25766, 8)

Unique areas: 26

Date range:
2006-12-23 00:00:00
2025-12-19 00:00:00

Missing values:


year                     0
week                     0
week_start               0
week_end                 0
area                     0
dengue_cases             0
week_length_days         0
reporting_period_days    0
dtype: int64


Duplicate area-week records: 0

Reporting-period lengths:


reporting_period_days
6       26
7    25714
8       26
Name: count, dtype: int64

In [46]:
processed_folder = project_root / "data" / "processed"
processed_folder.mkdir(parents=True, exist_ok=True)

output_path = (
    processed_folder / "dengue_weekly_cleaned.csv"
)

dengue_clean.to_csv(output_path, index=False)

print("Cleaned dengue dataset saved:")
print(output_path)

Cleaned dengue dataset saved:
c:\Users\Nethmi\Documents\GitHub\RiskWeave\data\processed\dengue_weekly_cleaned.csv


In [47]:
dengue_clean = dengue_clean.drop(
    columns=["week_length_days"]
)

print(dengue_clean.columns.tolist())

['year', 'week', 'week_start', 'week_end', 'area', 'dengue_cases', 'reporting_period_days']


In [48]:
output_path = (
    processed_folder / "dengue_weekly_cleaned.csv"
)

dengue_clean.to_csv(output_path, index=False)

print("Dataset saved successfully!")
print("Location:", output_path)
print("Shape:", dengue_clean.shape)

Dataset saved successfully!
Location: c:\Users\Nethmi\Documents\GitHub\RiskWeave\data\processed\dengue_weekly_cleaned.csv
Shape: (25766, 7)
